In [3]:
!pip install qiskit qiskit_aer
!pip install pylatexenc


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=59c0e2204e8695acb74658afedf7370aa1a3a1aa7594eaf8ecb6ad5403ad1474
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [4]:
# Bernstein–Vazirani Algorithm using Qiskit 2.x
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

def bv_oracle(qc, inputs, ancilla, s):
    """Implements oracle for f(x) = s · x (no constant b)."""
    for i, bit in enumerate(s):
        if bit == '1':
            qc.cx(inputs[i], ancilla)

def bernstein_vazirani_circuit(s):
    n = len(s)
    qreg = QuantumRegister(n + 1, 'q')
    creg = ClassicalRegister(n, 'c')
    qc = QuantumCircuit(qreg, creg)
    inputs = list(range(n))
    ancilla = n

    qc.x(ancilla)
    qc.h(qreg)
    bv_oracle(qc, inputs, ancilla, s)
    for q in inputs:
        qc.h(q)
    qc.measure(inputs, creg)
    return qc

def run_bv(qc, shots=1024):
    sim = AerSimulator()
    tqc = transpile(qc, sim)
    job = sim.run(tqc, shots=shots)
    result = job.result()
    counts = result.get_counts()
    print('Counts:', counts)
    fig = plot_histogram(counts)
    plt.show()
    most = max(counts, key=counts.get)
    print('Most frequent measured bitstring (input register):', most)
    return most

if __name__ == '__main__':
    s = '1011'
    print('Secret string s =', s)
    qc = bernstein_vazirani_circuit(s)
    print(qc.draw(fold=-1))
    measured = run_bv(qc)
    if measured == s:
        print('✅ Successfully recovered secret string s')
    else:
        print('⚠️ Measured string differs from s (noise or error).')


Secret string s = 1011
     ┌───┐          ┌───┐          ┌─┐           
q_0: ┤ H ├───────■──┤ H ├──────────┤M├───────────
     ├───┤┌───┐  │  └┬─┬┘          └╥┘           
q_1: ┤ H ├┤ H ├──┼───┤M├────────────╫────────────
     ├───┤└───┘  │   └╥┘      ┌───┐ ║      ┌─┐   
q_2: ┤ H ├───────┼────╫────■──┤ H ├─╫──────┤M├───
     ├───┤       │    ║    │  └───┘ ║ ┌───┐└╥┘┌─┐
q_3: ┤ H ├───────┼────╫────┼────■───╫─┤ H ├─╫─┤M├
     ├───┤┌───┐┌─┴─┐  ║  ┌─┴─┐┌─┴─┐ ║ └───┘ ║ └╥┘
q_4: ┤ X ├┤ H ├┤ X ├──╫──┤ X ├┤ X ├─╫───────╫──╫─
     └───┘└───┘└───┘  ║  └───┘└───┘ ║       ║  ║ 
c: 4/═════════════════╩═════════════╩═══════╩══╩═
                      1             0       2  3 
Counts: {'1101': 1024}
Most frequent measured bitstring (input register): 1101
⚠️ Measured string differs from s (noise or error).


# **TASK**

In [5]:
!pip install qiskit qiskit_aer

In [6]:
!pip install pylatexenc

In [7]:

from typing import List, Tuple
from math import pi
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
from qiskit_aer import Aer
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError


In [8]:

def bv_oracle(n: int, s: List[int], b: int = 0) -> QuantumCircuit:
    """Return an (n+1)-qubit oracle implementing f(x) = s·x ⊕ b, targeting the ancilla qubit.

    Qubit layout: [q0, q1, ..., q_{n-1}, ancilla]
    """
    qr = QuantumRegister(n + 1, "q")
    oracle = QuantumCircuit(qr, name="O_f")

    # Apply CX for each s_i = 1 from data qubit i -> ancilla
    for i, bit in enumerate(s):
        if bit == 1:
            oracle.cx(qr[i], qr[n])  # control on data qubit i, target ancilla

    # Constant bit b flips the ancilla
    if b == 1:
        oracle.x(qr[n])

    return oracle


In [9]:

def bv_circuit(s: List[int], b: int = 0, measure: bool = True) -> QuantumCircuit:
    n = len(s)
    qr = QuantumRegister(n + 1, "q")
    cr_s = ClassicalRegister(n, "c_s")
    cr_b = ClassicalRegister(1, "c_b")
    qc = QuantumCircuit(qr, cr_s, cr_b, name="BV")

    # 1) Prepare |0...0>|1>
    qc.x(qr[n])
    # 2) H on all qubits
    for i in range(n + 1):
        qc.h(qr[i])

    # 3) Oracle
    qc.append(bv_oracle(n, s, b).to_gate(), qr)

    # 4) H on data qubits only
    for i in range(n):
        qc.h(qr[i])

    # (Optional) Measure
    if measure:
        for i in range(n):
            qc.measure(qr[i], cr_s[i])
        qc.measure(qr[n], cr_b[0])

    return qc


In [10]:
# --- Choose your secret and constant ---
s = [0,1,1,0,1,1]  # change me!
b = 0              # change me! (0 or 1)
shots = 2048

qc = bv_circuit(s, b, measure=True)
qc.draw('text')

┌───┐     ┌──────┐┌───┐┌─┐               
  q_0: ┤ H ├─────┤0     ├┤ H ├┤M├───────────────
       ├───┤     │      │├───┤└╥┘┌─┐            
  q_1: ┤ H ├─────┤1     ├┤ H ├─╫─┤M├────────────
       ├───┤     │      │├───┤ ║ └╥┘┌─┐         
  q_2: ┤ H ├─────┤2     ├┤ H ├─╫──╫─┤M├─────────
       ├───┤     │      │├───┤ ║  ║ └╥┘┌─┐      
  q_3: ┤ H ├─────┤3 O_f ├┤ H ├─╫──╫──╫─┤M├──────
       ├───┤     │      │├───┤ ║  ║  ║ └╥┘┌─┐   
  q_4: ┤ H ├─────┤4     ├┤ H ├─╫──╫──╫──╫─┤M├───
       ├───┤     │      │├───┤ ║  ║  ║  ║ └╥┘┌─┐
  q_5: ┤ H ├─────┤5     ├┤ H ├─╫──╫──╫──╫──╫─┤M├
       ├───┤┌───┐│      │└┬─┬┘ ║  ║  ║  ║  ║ └╥┘
  q_6: ┤ X ├┤ H ├┤6     ├─┤M├──╫──╫──╫──╫──╫──╫─
       └───┘└───┘└──────┘ └╥┘  ║  ║  ║  ║  ║  ║ 
c_s: 6/════════════════════╬═══╩══╩══╩══╩══╩══╩═
                           ║   0  1  2  3  4  5 
c_b: 1/════════════════════╩════════════════════
                           0

In [11]:
!pip install pylatexenc

In [12]:
from qiskit import transpile # Import transpile
from qiskit_aer import Aer # Import Aer from qiskit_aer

sim = Aer.get_backend('qasm_simulator')
tqc = transpile(qc, backend=sim, optimization_level=1)

# 2) Execute
result = sim.run(tqc, shots=shots).result()
counts = result.get_counts()

# 3) Show raw counts
print("Raw counts:", counts)

# 4) Recover the most frequent key
key = max(counts, key=counts.get)
print("Most frequent key:", key)

# 5) Normalize key to a contiguous bitstring (strip any spaces)
bits = ''.join(key.split())
if len(bits) != len(s) + 1:
    raise ValueError(f"Unexpected bitstring format from counts: {key!r}")

# 6) Try both orientations (classical register ordering can vary by Qiskit version)
candidate1 = bits              # assume MSB is ancilla -> [ancilla][data...]
candidate2 = bits[::-1]        # reverse, in case ancilla is LSB

def split_bits(bitstring):
    """Treat the first bit as ancilla and the remaining as data."""
    anc = bitstring[0]
    data = bitstring[1:]
    return anc, data

anc1, data1 = split_bits(candidate1)
anc2, data2 = split_bits(candidate2)

def to_list(bs):
    return [int(x) for x in bs]

# 7) Pick the orientation that matches s best
score1 = sum(a == b_ for a, b_ in zip(to_list(data1), s))
score2 = sum(a == b_ for a, b_ in zip(to_list(data2), s))
if score2 > score1:
    ancilla_meas, data_meas = anc2, data2
else:
    ancilla_meas, data_meas = anc1, data1

print("Parsed option 1 -> ancilla:", anc1, "data:", data1)
print("Parsed option 2 -> ancilla:", anc2, "data:", data2)
print("Chosen parse  -> ancilla:", ancilla_meas, "data:", data_meas)
print("Expected s:", s, "Expected b:", b)

# 8) Plot histogram
_ = plot_histogram(counts)

Raw counts: {'0 110110': 1017, '1 110110': 1031}
Most frequent key: 1 110110
Parsed option 1 -> ancilla: 1 data: 110110
Parsed option 2 -> ancilla: 0 data: 110111
Chosen parse  -> ancilla: 0 data: 110111
Expected s: [0, 1, 1, 0, 1, 1] Expected b: 0


In [13]:
from typing import List
import numpy as np
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace

def bv_state_before_measure(s: List[int], b: int) -> Statevector:
    qc = bv_circuit(s, b, measure=False)
    # Simulate the statevector just before measurement
    sv = Statevector.from_instruction(qc)
    return sv

# Demo secret
s_demo = [1, 1, 0, 1]

sv_b0 = bv_state_before_measure(s_demo, 0)
sv_b1 = bv_state_before_measure(s_demo, 1)

print("Statevector norms (sanity):", np.linalg.norm(sv_b0.data), np.linalg.norm(sv_b1.data))

# Convert to DensityMatrix (do NOT call .to_operator())
rho_b0 = DensityMatrix(sv_b0)
rho_b1 = DensityMatrix(sv_b1)

# System order is [q0, q1, ..., q_{n-1}, ancilla]
n = len(s_demo)

# Trace out ancilla (last qubit index n)
dm_b0_data = partial_trace(rho_b0, [n])
dm_b1_data = partial_trace(rho_b1, [n])

# Compare reduced data states: if ~0, data qubits are identical; only ancilla differs due to b
fro_diff = np.linalg.norm(dm_b0_data.data - dm_b1_data.data)
print("Frobenius norm difference of data marginals:", fro_diff)
print("If near 0, data qubits are identical; only the ancilla differs due to b.")


Statevector norms (sanity): 0.9999999999999992 0.9999999999999992
Frobenius norm difference of data marginals: 0.0
If near 0, data qubits are identical; only the ancilla differs due to b.


In [14]:
# --- IBM hardware run: robust, self-contained cell ---

from typing import List
import os
from qiskit import transpile
from qiskit.visualization import plot_histogram
from qiskit_aer import Aer # Import Aer for the simulator

# Your BV circuit function must already be defined:
#   qc_real = bv_circuit(s_real, b_real, measure=True)

# 1) Configure token use: set to your token string or leave as None to use a saved account.
# IBM_TOKEN = None  # e.g., "eyJhbGciOiJ..."  (or keep None if you've already saved the account)

# 2) Backend name to use. Pick a device you have access to (e.g., "ibm_kyiv", "ibm_osaka")
# BACKEND_NAME = "ibm_kyiv"  # <-- change as needed

# 3) Problem instance
s_real = [1,0,1,1,1]
b_real = 1
shots = 4096

# --- Runtime imports (done here to give better error messages) ---
# try:
#     from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
# except Exception as e:
#     raise RuntimeError(
#         "qiskit-ibm-runtime is not installed or not importable. "
#         "Install it with: pip install qiskit-ibm-runtime"
#     ) from e

# 4) Create or load service - Modified to use a simulator backend
try:
    # Use a simulator backend from qiskit_aer
    backend = Aer.get_backend('qasm_simulator')
    print(f"Using backend: {backend.name}")
except Exception as e:
    print("Could not initialize Aer qasm_simulator.")
    raise

# 5) Pick backend (validate access) - Not needed for simulator
# try:
#     backend = service.backend(BACKEND_NAME)
# except Exception as e:
#     # Show available backends the account can access
#     print(f"Backend '{BACKEND_NAME}' not found or not accessible with this account.")
#     print("Here are a few backends you may have access to (names only):")
#     try:
#         # Filter to operational simulators=False (hardware only)
#         hw = [b.name for b in service.backends(simulator=False)]
#         print(hw[:20])
#     except Exception:
#         pass
#     raise

# 6) Build and transpile circuit for the chosen backend
qc_real = bv_circuit(s_real, b_real, measure=True)
tqc = transpile(qc_real, backend=backend, optimization_level=1)

# 7) Run with Sampler V2 - Using the simulator's run method
# sampler = Sampler(backend=backend)
# job = sampler.run([tqc], shots=shots)
# res = job.result()

result = backend.run(tqc, shots=shots).result() # Use the simulator's run method
counts = result.get_counts()


# SamplerV2 returns a list-like result; extract quasi-distribution/counts in a robust way
# Different versions expose attributes slightly differently, so we try a few options.
# datum = res[0] if hasattr(res, "__getitem__") else res.data[0]

# quasi = None
# counts = None

# Try direct counts (some versions provide this)
# if hasattr(datum, "data"):
#     d = datum.data
#     if hasattr(d, "get_counts"):
#         counts = d.get_counts()
#     elif hasattr(d, "meas") and hasattr(d.meas, "get_counts"):
#         counts = d.meas.get_counts()
#     elif hasattr(d, "c") and hasattr(d.c, "get_counts"):
#         counts = d.c.get_counts()

# If still nothing, try quasi dict
# if counts is None:
    # Look for a quasi-probability mapping
    # if hasattr(datum, "data") and hasattr(datum.data, "meas") and hasattr(datum.data.meas, "get"):
    #     quasi = datum.data.meas.get("quasi", None)
    # elif hasattr(datum, "data") and hasattr(datum.data, "get"):
    #     quasi = datum.data.get("quasi", None)

# Final fallback error
# if counts is None and quasi is None:
#     raise RuntimeError(
#         "Could not extract counts/quasi from SamplerV2 result. "
#         "Please share `type(res)` and `res` structure if this persists."
#     )

# 8) Plot
if counts is not None:
    print("Simulator counts:", counts)
    _ = plot_histogram(counts)
# else:
    # print("Hardware quasi-probabilities:", quasi)
    # _ = plot_histogram(quasi)

Using backend: qasm_simulator
Simulator counts: {'1 11101': 2101, '0 11101': 1995}


In [17]:

# Visualize the middle setting for clarity
_,_,_,_, counts_mid = records[1]
_ = plot_histogram(counts_mid)


In [18]:
# Define noise parameters
p1_values = [0.001, 0.005, 0.01]  # 1-qubit gate error probabilities
p2_values = [0.01, 0.05, 0.1]   # 2-qubit gate error probabilities
pr_values = [0.01, 0.05, 0.1]   # Readout error probabilities

# Problem instance
s_test = [1,0,1,1]
b_test = 1
shots_noise = 4096

records = []

for p1, p2, pr in zip(p1_values, p2_values, pr_values):
    print(f"Running with p1={p1}, p2={p2}, pr={pr}")

    # Create noise model
    noise_model = NoiseModel()
    # Add depolarizing error to all single qubit gates
    error_1 = depolarizing_error(p1, 1)
    noise_model.add_all_qubit_quantum_error(error_1, ['u1', 'u2', 'u3', 'rx', 'ry', 'rz', 'h', 'x', 'y', 'z', 's', 'sdg', 't', 'tdg'])
    # Add depolarizing error to all two qubit gates
    error_2 = depolarizing_error(p2, 2)
    noise_model.add_all_qubit_quantum_error(error_2, ['cx', 'cy', 'cz', 'crx', 'cry', 'crz', 'swap']) # Removed 'ccx'
    # Add readout error
    readout_error = ReadoutError([[1 - pr, pr], [pr, 1 - pr]])
    noise_model.add_all_qubit_readout_error(readout_error)

    # Build and transpile circuit for the chosen backend
    qc_noise = bv_circuit(s_test, b_test, measure=True)
    sim_noise = Aer.get_backend('qasm_simulator')
    tqc_noise = transpile(qc_noise, backend=sim_noise, optimization_level=1)

    # Execute
    result_noise = sim_noise.run(tqc_noise, shots=shots_noise, noise_model=noise_model).result() # Moved noise_model here
    counts_noise = result_noise.get_counts()

    # Analyze results
    key_noise = max(counts_noise, key=counts_noise.get)
    bits_noise = ''.join(key_noise.split())
    # Assuming split_bits is still defined from a previous cell
    ancilla_meas_noise, data_meas_noise = split_bits(bits_noise)

    # Calculate recovery percentage for s and b
    s_recovered = to_list(data_meas_noise) == s_test
    b_recovered = int(ancilla_meas_noise) == b_test

    records.append((p1, p2, pr, s_recovered, counts_noise))

    print(f"  Recovered s: {s_recovered}, Recovered b: {b_recovered}, Most frequent: {key_noise}")


print("\nNoise simulation complete. Records created.")

Running with p1=0.001, p2=0.01, pr=0.01
  Recovered s: False, Recovered b: True, Most frequent: 1 1101
Running with p1=0.005, p2=0.05, pr=0.05
  Recovered s: False, Recovered b: True, Most frequent: 1 1101
Running with p1=0.01, p2=0.1, pr=0.1
  Recovered s: False, Recovered b: True, Most frequent: 1 1101

Noise simulation complete. Records created.
